# TUM-VIE: Kalman-Style vs Recurrent Fusion — Deep-Dive

## Research Question
How do the **prediction–correction** and **recurrent-integration** fusion paradigms compare
on visual–inertial odometry in terms of accuracy, efficiency, and internal representations?

## Architecture Summary
| Architecture | Core computation | Key signature |
|---|---|---|
| `VisualIMURecurrentFusionBlock` | RLIF-gated recurrent cell | temporal integration of latents |
| `KalmanStyleSpikingFusionSurrogate` | IMU-predicted latent → innovation → correction | differentiable Kalman loop |

## Studies in This Notebook
1. **State/hidden-dim ablation** — sweep `hidden_dim` / `latent_dim` ∈ {32, 64, 128}
2. **Full training benchmark** on `mocap-6dof` with convergence curves
3. **Innovation-norm analysis** — how much does visual measurement correct IMU prediction?
4. **Spike-raster visualisation** — which timesteps fire? how bursty vs. tonic?
5. **Efficiency table** — RMSE / latency / params / spike-rate tradeoff

---
*All results persisted to `experiments/quant_pipeline/pipeline_runs.jsonl`.*

In [ ]:
from __future__ import annotations
from pathlib import Path
import time, json, datetime
from collections import defaultdict

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import optax
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tonic import transforms
from tonic.datasets import TUMVIE

import spyx.models as fm

plt.rcParams.update({"figure.dpi": 120, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

SEED           = 42
RECORDING      = "mocap-6dof"
SPATIAL_FACTOR = 0.2
N_TIME_BINS    = 20
N_SAMPLES      = 150
BATCH_SIZE     = 8
EPOCHS_SWEEP   = 15
EPOCHS_FULL    = 25
LR             = 2e-3
OUTPUT_DIM     = 3
DIM_SWEEP      = [32, 64, 128]

data_root    = Path("../data").resolve()
FIG_DIR      = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)
RESULTS_FILE = Path("../../experiments/quant_pipeline/pipeline_runs.jsonl").resolve()
ARCH_COLORS  = {"Recurrent": "#fc8d62", "Kalman": "#8da0cb"}

sensor       = TUMVIE.sensor_size
sensor_small = (int(sensor[0] * SPATIAL_FACTOR),
                int(sensor[1] * SPATIAL_FACTOR),
                sensor[2])
tfm = transforms.Compose([
    transforms.Downsample(spatial_factor=SPATIAL_FACTOR),
    transforms.ToFrame(sensor_size=sensor_small, n_time_bins=N_TIME_BINS),
])
ds = TUMVIE(save_to=str(data_root), recording=RECORDING, transform=tfm)
print(f"Sensor (original): {sensor}  ->  {sensor_small}")
print(f"Dataset size: {len(ds):,}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Data Loading + Helpers
# ─────────────────────────────────────────────────────────────────────────────
def build_arrays(limit: int = N_SAMPLES):
    xs, us, ys = [], [], []
    n = min(limit, len(ds))
    for i in range(n):
        d, t = ds[i]
        left  = np.asarray(d["events_left"],  dtype=np.float32)
        right = np.asarray(d["events_right"], dtype=np.float32)
        x     = np.concatenate([left, right], axis=1)
        imu   = np.asarray(d["imu"], dtype=np.float32)
        imu   = imu.mean(axis=0) if (imu.ndim == 2 and imu.shape[0] > 0) \
                else np.zeros(6, np.float32)
        mocap = np.asarray(t["mocap"], dtype=np.float32)
        mocap = mocap[-1] if mocap.ndim == 2 else mocap
        xs.append(x); us.append(imu); ys.append(mocap[:3])
    return np.stack(xs), np.stack(us), np.stack(ys)

print("Building arrays ...")
X_all, U_all, Y_all = build_arrays()
N_TOTAL, T_BINS, H_SPATIAL, W_SPATIAL, N_CHANNELS = X_all.shape
INPUT_HW = (H_SPATIAL, W_SPATIAL); INPUT_CHANNELS = N_CHANNELS; IMU_DIM = 6
print(f"X: {X_all.shape}  U: {U_all.shape}  Y: {Y_all.shape}")

idx_tr, idx_te = train_test_split(np.arange(N_TOTAL), test_size=0.2, random_state=SEED)
idx_tr, idx_va = train_test_split(idx_tr, test_size=0.125, random_state=SEED + 1)

def _j(a, i): return jnp.asarray(a[i])
Xtr_j, Utr_j, Ytr_j = _j(X_all,idx_tr), _j(U_all,idx_tr), jnp.asarray(Y_all[idx_tr])
Xva_j, Uva_j, Yva_j = _j(X_all,idx_va), _j(U_all,idx_va), jnp.asarray(Y_all[idx_va])
Xte_j, Ute_j, Yte_j = _j(X_all,idx_te), _j(U_all,idx_te), jnp.asarray(Y_all[idx_te])

Y_mean = float(Y_all[idx_tr].mean()); Y_std = float(Y_all[idx_tr].std()) + 1e-6
def _z(Y): return jnp.asarray((np.asarray(Y) - Y_mean) / Y_std)
Ytr_z, Yva_z, Yte_z = _z(Ytr_j), _z(Yva_j), _z(Yte_j)

def _tm(X, idx): return jnp.swapaxes(X[idx], 0, 1)
def make_seqs(u, T): return jnp.broadcast_to(u[None], (T, *u.shape))

def make_batch_kv(X, U, Y, bs, key):
    idx_b = jax.random.choice(key, X.shape[0], shape=(bs,), replace=False)
    return _tm(X, idx_b), U[idx_b], Y[idx_b]

def iter_batches_kv(X, U, Y):
    n = (X.shape[0] // BATCH_SIZE) * BATCH_SIZE
    for b in range(0, n, BATCH_SIZE):
        yield _tm(X, np.arange(b, b + BATCH_SIZE)), U[b:b+BATCH_SIZE], Y[b:b+BATCH_SIZE]

print(f"train: {len(idx_tr)} | val: {len(idx_va)} | test: {len(idx_te)}")


# ─────────────────────────────────────────────────────────────────────────────
# 2. Model Builders (parameterised by dim)
# ─────────────────────────────────────────────────────────────────────────────
def build_recurrent(hidden_dim: int):
    vis_cfg = fm.ConvConfig(input_hw=INPUT_HW, input_channels=INPUT_CHANNELS,
                             channels1=24, channels2=48, output_dim=hidden_dim)
    cfg = fm.VisualIMURecurrentConfig(vision_cfg=vis_cfg, imu_dim=IMU_DIM,
                                       traj_dim=OUTPUT_DIM, hidden_dim=hidden_dim,
                                       output_dim=OUTPUT_DIM)
    def fwd(x_seq, imu):
        T = x_seq.shape[0]; B = x_seq.shape[1]
        return fm.VisualIMURecurrentFusionBlock(cfg)(
            x_seq, make_seqs(imu, T), jnp.zeros((T, B, OUTPUT_DIM)))
    return hk.without_apply_rng(hk.transform(lambda x, u: fwd(x, u)))


def build_kalman(latent_dim: int):
    vis_cfg = fm.ConvConfig(input_hw=INPUT_HW, input_channels=INPUT_CHANNELS,
                             channels1=24, channels2=48, output_dim=latent_dim)
    kal_cfg = fm.KalmanFusionConfig(latent_dim=latent_dim, output_dim=OUTPUT_DIM)
    def fwd(x_seq, imu):
        _, vis_aux = fm.ConvLIFSNN(vis_cfg)(x_seq)
        vis_seq = vis_aux["logits_seq"]
        return fm.KalmanStyleSpikingFusionSurrogate(kal_cfg)(vis_seq, make_seqs(imu, vis_seq.shape[0]))
    return hk.without_apply_rng(hk.transform(lambda x, u: fwd(x, u)))


# ─────────────────────────────────────────────────────────────────────────────
# 3. Training Utility
# ─────────────────────────────────────────────────────────────────────────────
def count_params(params) -> int:
    return int(sum(x.size for x in jax.tree_util.tree_leaves(params)))


def run_training(transformed, epochs: int, label: str = "") -> dict:
    key0 = jax.random.PRNGKey(SEED)
    xb0, ub0, _ = make_batch_kv(Xtr_j, Utr_j, Ytr_z, BATCH_SIZE, key0)
    params    = transformed.init(key0, xb0, ub0)
    n_params  = count_params(params)
    opt       = optax.adamw(LR, weight_decay=1e-4)
    opt_state = opt.init(params)

    @jax.jit
    def step(params, opt_state, xb, ub, yb):
        def loss_fn(p):
            pred, aux = transformed.apply(p, xb, ub)
            return jnp.mean((pred - yb)**2), (pred, aux)
        (loss, (pred, aux)), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
        updates, new_opt = opt.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_opt, loss, aux

    @jax.jit
    def infer(params, xb, ub): return transformed.apply(params, xb, ub)

    history = defaultdict(list)
    key     = jax.random.PRNGKey(SEED + 13)
    for epoch in range(1, epochs + 1):
        key, sk = jax.random.split(key)
        xb, ub, yb = make_batch_kv(Xtr_j, Utr_j, Ytr_z, BATCH_SIZE, sk)
        params, opt_state, tr_mse, tr_aux = step(params, opt_state, xb, ub, yb)
        va_maes = []
        for xvb, uvb, yvb in iter_batches_kv(Xva_j, Uva_j, Yva_z):
            pv, _ = infer(params, xvb, uvb)
            va_maes.append(np.mean(np.abs(np.asarray(pv) - np.asarray(yvb)), axis=0))
        va_rmse = float(np.sqrt(np.mean(np.mean(va_maes, axis=0)**2))) if va_maes else 0.0
        sr      = float(np.mean(np.asarray(tr_aux.get("spike_rate", jnp.zeros(1)))))
        innov   = float(np.asarray(tr_aux.get("innovation_norm", jnp.zeros(1))))
        history["epoch"].append(epoch); history["train_mse"].append(float(tr_mse))
        history["val_rmse"].append(va_rmse); history["spike_rate"].append(sr)
        history["innovation_norm"].append(innov)

    te_preds = []
    for xtb, utb, _ in iter_batches_kv(Xte_j, Ute_j, Yte_z):
        pt, _ = infer(params, xtb, utb); te_preds.append(np.asarray(pt))
    te_z    = np.concatenate(te_preds, axis=0)
    te_m    = te_z  * Y_std + Y_mean
    gt_m    = np.asarray(Yte_j[:te_z.shape[0]]) * Y_std + Y_mean
    te_rmse = float(np.sqrt(np.mean((te_m - gt_m)**2)))

    infer(params, xb0, ub0)
    t0 = time.perf_counter()
    for _ in range(50): jax.block_until_ready(infer(params, xb0, ub0))
    latency_ms = (time.perf_counter() - t0) / 50 * 1000

    xr, ur, _ = make_batch_kv(Xte_j, Ute_j, Yte_z, min(BATCH_SIZE, Xte_j.shape[0]),
                               jax.random.PRNGKey(100))
    _, raster_aux = infer(params, xr, ur)
    return {
        "history":    dict(history),  "test_rmse":  te_rmse,
        "n_params":   n_params,        "latency_ms": latency_ms,
        "spike_rate": history["spike_rate"][-1],
        "raster_aux": {k: np.asarray(v) for k, v in raster_aux.items()},
        "te_m": te_m, "gt_m": gt_m, "label": label,
    }


# ─────────────────────────────────────────────────────────────────────────────
# 4. Dimension Sweep
# ─────────────────────────────────────────────────────────────────────────────
sweep_results = {"Recurrent": {}, "Kalman": {}}
print("Running dimension sweep ...")
for dim in DIM_SWEEP:
    for arch, builder in [("Recurrent", build_recurrent), ("Kalman", build_kalman)]:
        label = f"{arch}-d{dim}"
        print(f"  {label} ...", end=" ", flush=True)
        res = run_training(builder(dim), EPOCHS_SWEEP, label=label)
        sweep_results[arch][dim] = res
        print(f"test_rmse={res['test_rmse']:.4f}  params={res['n_params']:,}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. Full Training — Best Dim per Architecture
# ─────────────────────────────────────────────────────────────────────────────
best_dims = {
    arch: min(DIM_SWEEP,
              key=lambda d: sweep_results[arch][d]["history"]["val_rmse"][-1])
    for arch in ("Recurrent", "Kalman")
}
print("Best dims:", best_dims)

full_results = {}
for arch, builder in [("Recurrent", build_recurrent), ("Kalman", build_kalman)]:
    dim = best_dims[arch]
    print(f"\n-- Full training: {arch} (dim={dim}) --")
    full_results[arch] = run_training(builder(dim), EPOCHS_FULL,
                                      label=f"{arch}-full-d{dim}")
    r = full_results[arch]
    print(f"  test_rmse={r['test_rmse']:.4f}  params={r['n_params']:,}  "
          f"latency={r['latency_ms']:.1f}ms  spike_rate={r['spike_rate']:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 1 – Dim Sweep + Full Training Curves
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for arch, color in ARCH_COLORS.items():
    rmses  = [sweep_results[arch][d]["test_rmse"]  for d in DIM_SWEEP]
    params = [sweep_results[arch][d]["n_params"]   for d in DIM_SWEEP]
    spikes = [sweep_results[arch][d]["spike_rate"] for d in DIM_SWEEP]
    axes[0,0].plot(DIM_SWEEP, rmses, "o-", label=arch, color=color, lw=2)
    axes[0,1].scatter([np.log10(p) for p in params], rmses,
                      s=[50 + 200*s for s in spikes],
                      label=arch, color=color, alpha=0.8, edgecolors="white")
    for dim, p, r in zip(DIM_SWEEP, params, rmses):
        axes[0,1].annotate(f"d={dim}", (np.log10(p), r),
                            textcoords="offset points", xytext=(5, 0), fontsize=7)
    axes[0,2].plot(DIM_SWEEP, spikes, "s--", label=arch, color=color, lw=1.5)

axes[0,0].set_xlabel("Hidden / Latent Dim"); axes[0,0].set_ylabel("Test RMSE (m)")
axes[0,0].set_title("Dim Sensitivity", fontweight="bold"); axes[0,0].legend()
axes[0,1].set_xlabel("log10(Params)"); axes[0,1].set_ylabel("Test RMSE (m)")
axes[0,1].set_title("Params vs RMSE (dot size proportional to spike rate)", fontweight="bold")
axes[0,1].legend()
axes[0,2].set_xlabel("Dim"); axes[0,2].set_ylabel("Spike Rate")
axes[0,2].set_title("Spike Rate vs Dim", fontweight="bold"); axes[0,2].legend()

for arch, color in ARCH_COLORS.items():
    h  = full_results[arch]["history"]
    ep = h["epoch"]
    axes[1,0].plot(ep, h["train_mse"],        label=arch, color=color, lw=2)
    axes[1,1].plot(ep, h["val_rmse"],         label=arch, color=color, lw=2)
    axes[1,2].plot(ep, h["innovation_norm"],  label=arch, color=color, lw=2)

axes[1,0].set_title("Train MSE",       fontweight="bold"); axes[1,0].set_xlabel("Epoch")
axes[1,1].set_title("Val RMSE (m)",    fontweight="bold"); axes[1,1].set_xlabel("Epoch")
axes[1,2].set_title("Innovation Norm", fontweight="bold"); axes[1,2].set_xlabel("Epoch")
for ax in axes[1]: ax.legend(fontsize=9)

fig.suptitle(f"Kalman vs Recurrent Fusion -- {RECORDING}", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "kalman_vs_recurrent_overview.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 2 – Trajectory Prediction Comparison
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 7), squeeze=False)

for row, (arch, color) in enumerate(ARCH_COLORS.items()):
    te_m = full_results[arch]["te_m"]
    gt_m = full_results[arch]["gt_m"]
    n    = min(te_m.shape[0], gt_m.shape[0])
    for col, axis_label in enumerate(["tx (m)", "ty (m)", "tz (m)"]):
        ax = axes[row][col]
        ax.plot(gt_m[:n, col], color="#555555", lw=1,   label="Ground Truth", alpha=0.8)
        ax.plot(te_m[:n, col], color=color,    lw=1.4, label=arch)
        if row == 0: ax.set_title(axis_label, fontweight="bold")
        if col == 0: ax.set_ylabel(arch, fontsize=9)
        if row == 1: ax.set_xlabel("sample index")
        if row == 0 and col == 0: ax.legend(fontsize=7)

fig.suptitle(f"Predicted vs GT Translation -- test split ({RECORDING})", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "kalman_vs_recurrent_trajectories.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 3 – Innovation Norm Analysis (Kalman-specific) + Spike Dynamics
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

innov_hist = full_results["Kalman"]["history"]["innovation_norm"]
val_hist   = full_results["Kalman"]["history"]["val_rmse"]
ep         = full_results["Kalman"]["history"]["epoch"]

ax1, ax2 = axes[0], axes[0].twinx()
l1, = ax1.plot(ep, innov_hist, color="#8da0cb", lw=2, label="Innovation Norm")
l2, = ax2.plot(ep, val_hist,   color="#e41a1c", lw=1.5, ls="--", label="Val RMSE")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Innovation Norm (||visual - IMU pred||)", color="#8da0cb")
ax2.set_ylabel("Val RMSE (m)", color="#e41a1c")
axes[0].set_title("Kalman: Innovation vs Val RMSE over Training", fontweight="bold")
axes[0].legend(handles=[l1, l2], fontsize=8)

for arch, color in ARCH_COLORS.items():
    sr_hist = full_results[arch]["history"]["spike_rate"]
    ep_a    = full_results[arch]["history"]["epoch"]
    axes[1].plot(ep_a, sr_hist, label=arch, color=color, lw=2)

axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Mean Spike Rate")
axes[1].set_title("Spike Rate Dynamics over Training", fontweight="bold")
axes[1].legend(fontsize=9)

fig.suptitle("Kalman Innovation Analysis + Spike Dynamics", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "kalman_innovation_and_spikes.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 4 – Latent Activation Raster
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax_idx, (arch, color) in enumerate(ARCH_COLORS.items()):
    raux = full_results[arch]["raster_aux"]
    if "logits_seq" in raux:
        logits = raux["logits_seq"]
        if logits.ndim == 3:
            mean_seq = logits.mean(axis=1)
            binary   = (mean_seq > 0).astype(float)
            ax = axes[ax_idx]
            im = ax.imshow(binary.T, aspect="auto", cmap="Greys",
                           origin="lower", interpolation="nearest")
            ax.set_xlabel("Timestep (within window)")
            ax.set_ylabel("Latent Dimension")
            ax.set_title(f"{arch}: Latent Activation Raster (batch-mean, threshold=0)",
                         fontweight="bold")
            plt.colorbar(im, ax=ax, label="Active")
        else:
            axes[ax_idx].text(0.5, 0.5, "logits_seq not 3-D",
                               ha="center", va="center",
                               transform=axes[ax_idx].transAxes)
    else:
        available = list(raux.keys())
        axes[ax_idx].text(0.5, 0.5,
                           f"No logits_seq in aux\navailable: {available}",
                           ha="center", va="center",
                           transform=axes[ax_idx].transAxes)
        axes[ax_idx].set_title(f"{arch}: aux keys = {available}", fontweight="bold")

fig.suptitle("Latent Activation Raster (demo batch, test set)", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "kalman_vs_recurrent_raster.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Summary Table + Persist
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for arch in ("Recurrent", "Kalman"):
    for dim in DIM_SWEEP:
        r = sweep_results[arch][dim]
        rows.append({
            "arch": arch, "dim": dim,
            "sweep_rmse":  round(r["test_rmse"],   4),
            "spike_rate":  round(r["spike_rate"],  4),
            "n_params":    r["n_params"],
            "latency_ms":  round(r["latency_ms"],  2),
        })
for arch in ("Recurrent", "Kalman"):
    r = full_results[arch]
    rows.append({
        "arch": arch, "dim": f"{best_dims[arch]} (full)",
        "sweep_rmse": round(r["test_rmse"],  4),
        "spike_rate": round(r["spike_rate"], 4),
        "n_params":   r["n_params"],
        "latency_ms": round(r["latency_ms"], 2),
    })

df = pd.DataFrame(rows).sort_values(["arch", "sweep_rmse"])
print("-- Kalman vs Recurrent Benchmark --")
print(df.to_string(index=False))

run_record = {
    "timestamp":   datetime.datetime.utcnow().isoformat() + "Z",
    "experiment":  "tumvie_kalman_vs_recurrent",
    "recording":   RECORDING,
    "epochs_sweep":EPOCHS_SWEEP,
    "epochs_full": EPOCHS_FULL,
    "dim_sweep":   DIM_SWEEP,
    "best_dims":   best_dims,
    "sweep_results": {
        arch: {str(dim): {k: v for k, v in r.items()
               if k not in ("raster_aux", "te_m", "gt_m")}
               for dim, r in res_by_dim.items()}
        for arch, res_by_dim in sweep_results.items()
    },
    "full_results": {
        arch: {k: v for k, v in r.items()
               if k not in ("raster_aux", "te_m", "gt_m")}
        for arch, r in full_results.items()
    },
}
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_FILE, "a") as f:
    f.write(json.dumps(run_record) + "\n")
print(f"\nResults appended to {RESULTS_FILE}")